# Create AnnData objects for external cohort
##### Franziska Niemeyer

In [ ]:
import numpy
import os
import scanpy as sc
import numpy as np
import sys
import pandas as pd
import matplotlib.pyplot as plt

WORKING_DIR = "."
DATA_DIR = "data"   # Space Ranger output + annotations/ for the external cohort
OUT_DIR = os.path.join(WORKING_DIR, "figures")
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

sc.settings.figdir = OUT_DIR

##### Load data

In [ ]:
aggr_matrix = os.path.join(DATA_DIR, "filtered_feature_bc_matrix.h5")
adata = sc.read_10x_h5(aggr_matrix)
adata.var_names_make_unique()
adata

##### Load Spatial Data

In [ ]:
import scanpy as sc
import squidpy as sq
from PIL import Image
import os
import json

def load_aggregated_visium(
        aggr_path: str,
        barcode_suffix_to_sample: dict,
        spatial_base: str = None,
        library_id: str = None
):
    """
    Load aggregated Visium data and map sample names using barcode suffixes,
    then attach spatial images per sample using Squidpy.
    """
    adata = sc.read_visium(aggr_path, load_images=False)
    adata.var_names_make_unique()

    barcodes = adata.obs_names
    suffixes = [bc.split('-')[-1] for bc in barcodes]

    sample_names = []
    for suffix in suffixes:
        if suffix not in barcode_suffix_to_sample:
            raise ValueError(f"Suffix '{suffix}' not found in mapping.")
        sample_names.append(barcode_suffix_to_sample[suffix])

    adata.obs['sample'] = sample_names

    if spatial_base is None:
        spatial_base = os.path.join(aggr_path, "spatial")

    for sample in set(sample_names):
        sample_spatial_path = os.path.join(spatial_base, sample)
        tissue_img_path = os.path.join(sample_spatial_path, "tissue_hires_image.png")
        scalefactors_path = os.path.join(sample_spatial_path, "scalefactors_json.json")

        if not os.path.isfile(tissue_img_path) or not os.path.isfile(scalefactors_path):
            raise FileNotFoundError(f"Missing spatial image or scalefactors for {sample} under {sample_spatial_path}")

        img = Image.open(tissue_img_path)
        img_np = np.array(img)

        with open(scalefactors_path) as f:
            scale_factors = json.load(f)

        adata.uns[f"spatial"][sample] = {
            "images": {"hires": img_np},
            "scalefactors": scale_factors,
        }

    spatial_df = pd.DataFrame(np.full((adata.n_obs, 2), np.nan),
                              index=adata.obs_names,
                              columns=["x", "y"])

    for suffix, sample in barcode_suffix_to_sample.items():

        # Load correct file
        for filename in ["tissue_positions.csv", "tissue_positions_list.csv", "aggr_tissue_positions_list.csv", "aggr_tissue_positions.csv"]:
            positions_path = os.path.join(spatial_base, filename)
            if os.path.exists(positions_path):
                break
        else:
            raise FileNotFoundError(f"No tissue position file found for sample '{sample}'")

        coords = pd.read_csv(positions_path, header=None)
        coords.columns = ['barcode', 'in_tissue', 'array_row', 'array_col', 'pxl_row_in_fullres', 'pxl_col_in_fullres']

        if not filename in ["aggr_tissue_positions.csv", "aggr_tissue_positions_list.csv", "tissue_positions_list.csv"]:
            # coords['barcode_full'] = coords['barcode'].astype(str) + f"-{suffix}"
            coords['barcode_full'] = coords['barcode'].map(lambda x: x.replace('-1', f'-{suffix}'))
            coords.set_index('barcode_full', inplace=True)
        else:
            coords.set_index('barcode', inplace=True)

        coords = coords.loc[coords.index.intersection(spatial_df.index)]
        spatial_df.loc[coords.index] = coords[["pxl_col_in_fullres", "pxl_row_in_fullres"]].astype(int).values  # x, y

    adata.obsm["spatial"] = spatial_df.to_numpy(dtype=int)

    return adata

In [ ]:
barcode_suffix_to_sample = {
    '1': 'HC-TMA1',
    '2': 'HC-TMA2',
    '3': 'HC-TMA3',
    '4': 'HC-TMA4'
}

adata = load_aggregated_visium(
    aggr_path=os.path.join(DATA_DIR),
    barcode_suffix_to_sample=barcode_suffix_to_sample,
    spatial_base=os.path.join(DATA_DIR, "spatial")
)
adata.var_names_make_unique()

In [ ]:
adata

In [ ]:
anno_file = os.path.join(DATA_DIR, f"annotations/aggr/TissueType.csv")
patients_file = os.path.join(DATA_DIR, f"annotations/aggr/Patient-ids.csv")
outcome_file = os.path.join(DATA_DIR, f"annotations/aggr/OutcomeGroups.csv")

annos = lambda folder, name: pd.read_csv(folder, index_col=0, names=[name], header=0)
adata.obs = adata.obs.merge(how='left', right=annos(anno_file, 'histology'), left_index=True, right_index=True)
adata.obs = adata.obs.merge(how='left', right=annos(patients_file, 'patient'), left_index=True, right_index=True)
adata.obs = adata.obs.merge(how='left', right=annos(outcome_file, 'outcome'), left_index=True, right_index=True)

adata.obs['patient'] = adata.obs['patient'].replace({'OVA15-16': 'OVA15'})
adata.obs

##### Quality Control

In [ ]:
import seaborn as sns

sc.pp.calculate_qc_metrics(adata, inplace=True)
adata.var_names_make_unique()
sns.jointplot(
    data=adata.obs,
    x="total_counts",
    y="n_genes_by_counts",
    kind="hex",
)
plt.savefig(os.path.join(OUT_DIR, "total_counts_vs_n_genes.png"), dpi=600)
plt.show()

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# Plot histograms for recalculated QC metrics
fig, axs = plt.subplots(1, 2, figsize=(18, 5))

# Total counts
sns.histplot(adata.obs["total_counts"], kde=True, bins=100, ax=axs[0])
axs[0].set_title("Total Counts")
axs[0].set_xlabel("Counts")
axs[0].set_ylabel("Frequency")

# Number of genes by counts
sns.histplot(adata.obs["n_genes_by_counts"], kde=True, bins=100, ax=axs[1])
axs[1].set_title("Number of Genes by Counts")
axs[1].set_xlabel("Number of Genes")
axs[1].set_ylabel("Frequency")

# Adjust layout for better visualization
plt.tight_layout()
plt.show()

In [ ]:
import math

slides = adata.obs["sample"].unique()
n = len(slides)

ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes = axes.flatten()

for i, slide in enumerate(slides):
    sq.pl.spatial_scatter(
        adata[adata.obs["sample"] == slide],
        library_id=slide,
        color="total_counts",
        title=f"{slide} - Total counts",
        size=1.5,
        alpha=1,
        img_alpha=0.2,
        ax=axes[i],
    )

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "total_counts_spatial.pdf"))
plt.show()

In [ ]:
# Plot histogram and density of library sizes
plt.figure(figsize=(8, 6))
sns.histplot(adata.obs['total_counts'], kde=True, color='gray', label='Density')
plt.axvline(x=700, color='red', linestyle='--', label='Threshold (700)')
plt.title('Distribution of Library Sizes')
plt.xlabel('Library Size (Total Counts)')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
# Apply a threshold for low library size
lib_size_threshold = 150
adata.obs['qc_lib_size'] = adata.obs['total_counts'] < lib_size_threshold

# Count the number of spots flagged as low library size
low_library_count = adata.obs['qc_lib_size'].sum()
print(f"Number of spots with low library size: {low_library_count}")

# Make it categorical with readable labels
adata.obs["qc_lib_size_plot"] = (
    adata.obs["qc_lib_size"]
    .map({True: "low", False: "ok"})
    .astype("category")
)

In [ ]:
adata.obs

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

histology_order = sorted(adata.obs["histology"].dropna().unique())
adata.obs["histology"] = pd.Categorical(
    adata.obs["histology"],
    categories=histology_order,
    ordered=True
)

palette = sns.color_palette("muted", n_colors=len(histology_order))
cmap = ListedColormap(palette)

qc_lib_size_order = sorted(adata.obs["qc_lib_size_plot"].dropna().unique())

# Visualize spots flagged for low library size
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 6 * nrows))
axes = axes.flatten()

for i, slide in enumerate(slides):
    ad = adata[adata.obs["sample"] == slide].copy()

    # re-enforce category order after subsetting
    ad.obs["qc_lib_size_plot"] = ad.obs["qc_lib_size_plot"].cat.set_categories(["low", "ok"])
    
    sq.pl.spatial_scatter(
        ad,
        library_id=slide,
        color=["qc_lib_size_plot"],
        size=1.5,
        title=f"{slide}",
        img_alpha=.5,
        palette=ListedColormap(["blue", "lightgray"]),
        ax=axes[i],
        legend_loc=None
    )

    axes[i].set_axis_off()

handles = [
    Patch(facecolor=cmap(i), label=cat)
    for i, cat in enumerate(qc_lib_size_order)
]

fig.legend(
    handles=handles,
    loc="lower center",
    title="Library size",
    frameon=False
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(os.path.join(OUT_DIR, "total_counts_thresholding.pdf"))
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 6 * nrows))
axes = axes.flatten()

for i, slide in enumerate(slides):
    ad = adata[adata.obs["sample"] == slide].copy()

    # preserve all categories after subsetting
    ad.obs["histology"] = ad.obs["histology"].cat.set_categories(histology_order)

    sq.pl.spatial_scatter(
        ad,
        library_id=slide,
        color="histology",
        size=1.5,
        title=f"{slide}",
        palette=cmap,
        img_alpha=0.5,
        ax=axes[i],
        legend_loc=None
    )

    axes[i].set_axis_off()

handles = [
    Patch(facecolor=cmap(i), label=cat)
    for i, cat in enumerate(histology_order)
]

fig.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.1),
    title="Histology",
    frameon=False
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(os.path.join(OUT_DIR, "histology_spatial.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes = axes.flatten()

# fix global color scale
vmin = adata.obs["n_genes_by_counts"].min()
vmax = adata.obs["n_genes_by_counts"].max()

for i, slide in enumerate(slides):
    ad = adata[adata.obs["sample"] == slide]

    sq.pl.spatial_scatter(
        ad,
        library_id=slide,
        color="n_genes_by_counts",
        title=f"{slide}",
        size=1.5,
        alpha=1,
        img_alpha=0.2,
        vmin=vmin,
        vmax=vmax,
        ax=axes[i],
        colorbar=False
    )

    axes[i].set_axis_off()

# for j in range(i + 1, len(axes)):
#     axes[j].axis("off")

import matplotlib as mpl

norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
sm = mpl.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=axes,
    location="right",
    fraction=0.02,
    pad=0.02
)
cbar.set_label("Genes by Counts")

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.savefig(os.path.join(OUT_DIR, "n_genes_spatial.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Plot the distribution of expressed genes
plt.figure(figsize=(8, 6))
sns.histplot(adata.obs['n_genes_by_counts'], kde=True, color='gray')
plt.title('Distribution of Expressed Genes per Spot')
plt.xlabel('Number of Expressed Genes')
plt.ylabel('Density')
plt.axvline(x=100, color='red', linestyle='--', label='Threshold (600)')
plt.legend()
plt.show()

In [ ]:
# Apply the threshold for the number of expressed genes
threshold_expressed_genes = 100
adata.obs['qc_expressed_genes'] = adata.obs['n_genes_by_counts'] < threshold_expressed_genes

# Count the number of spots flagged as low-quality based on expressed genes
low_quality_genes_count = adata.obs['qc_expressed_genes'].sum()
print(f"Number of spots with fewer than {threshold_expressed_genes} expressed genes: {low_quality_genes_count}")

# Make it categorical with readable labels
adata.obs["qc_expressed_genes_plot"] = (
    adata.obs["qc_expressed_genes"]
    .map({True: "low", False: "ok"})
    .astype("category")
)

### Filtering

In [ ]:
print(f"Number of genes before filtering: {adata.n_vars}")
sc.pp.filter_genes(adata, min_cells=3)
print(f"Number of genes retained: {adata.n_vars}")

In [ ]:
adata.layers['counts'] = adata.X.copy()

In [ ]:
adata.write(os.path.join(WORKING_DIR, "adata_filtered_raw.h5ad"))

### Normalization

In [ ]:
adata.layers['library-nomalized'] = sc.pp.normalize_total(adata, target_sum=1e4, exclude_highly_expressed=True, copy=True).X
adata.layers['log-transformed'] = sc.pp.log1p(adata, copy=True).X

In [ ]:
adata.layers['scaled'] = adata.X.copy()
sc.pp.scale(adata, zero_center=True, layer='scaled')
sc.pp.filter_genes(adata.layers['scaled'], min_counts=1)

In [ ]:
original_counts = adata.layers['counts'].sum(axis=1)
normalized_counts = adata.layers['library-nomalized'].sum(axis=1)
scaled_counts = adata.layers['scaled'].sum(axis=1)
log_transformed_counts = adata.layers['log-transformed'].sum(axis=1)

plt.figure(figsize=(8, 3))
sns.histplot(original_counts.A1, color="blue", label="Before normalization", kde=True)
sns.histplot(normalized_counts.A1, color="orange", label="Library-normalized", kde=True)
sns.histplot(scaled_counts, color="green", label="Normalized and scaled", kde=True)
sns.histplot(log_transformed_counts, color="lightblue", label="Log-normalized", kde=True)
plt.legend()
plt.title(f'Effect of Normalization and Scaling')
plt.xlabel('Total Expression')
plt.ylabel('Number of Cells')

In [ ]:
adata.write_h5ad(os.path.join(WORKING_DIR, "adata.h5ad"))